In [3]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd 
import geopandas as gpd

In [ ]:
#On définit la police de caractères pour les graphiques
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Helvetica", "DejaVu Sans", "Arial"]

#On améliore la qualité des graphiques
%config InlineBackend.figure_format = 'retina'

In [ ]:
#Chargement du fond de carte de l'Île-de-France
gdf_iris = gpd.read_file("IDF_iris.geojson")
gdf_iris.head()

In [ ]:
#Récupération du tracé de la Ligne 14 (site IDFM)
ligne_14 = gpd.read_file("/Users/borhane/Documents/2A ENSAE/Projet Python 2A/projet_python_2a_ensae/traces-des-lignes-de-transport-en-commun-idfm-2.geojson")
ligne_14.head()

#On sélectionne la zone d'étude correspondant à la commune de Saint-Ouen (code INSEE 93070)
col_insee = 'insee_com' if 'insee_com' in gdf_iris.columns else 'depcom'
zone_etude = gdf_iris[gdf_iris[col_insee].astype(str) == '93070'] #Je force ici à convertir en str pour éviter les problèmes de type de données (les codes iNSEE sont lus comme des int ce qui pose problème)

# Vérifier et convertir si nécessaire le système de coordonnées de la ligne 14 pour qu'il corresponde à celui de la zone d'étude
if ligne_14.crs != zone_etude.crs:
    ligne_14 = ligne_14.to_crs(zone_etude.crs)

In [ ]:
#On récupère les emplacements des stations de métro de la ligne 14 (site IDFM)
gdf_stations = gpd.read_file("/Users/borhane/Documents/2A ENSAE/Projet Python 2A/projet_python_2a_ensae/emplacement-des-gares-idf.geojson")

#On vérifie que le système de coordonnées des stations correspond à celui de la zone d'étude, sinon on le convertit
if gdf_stations.crs != zone_etude.crs:
    gdf_stations = gdf_stations.to_crs(zone_etude.crs)

# Récupérer les limites géographiques (bords min et max) de Saint-Ouen
minx, miny, maxx, maxy = zone_etude.total_bounds #On récupère les limites géographiques de la zone d'étude (Saint-Ouen) pour ajuster le zoom du graphique (longitude et latitude)

marge_x = (maxx - minx) * 0.05
marge_y = (maxy - miny) * 0.05

#On sélectionne uniquement les stations de la 14 à Saint-Ouen
stations_st_ouen = gdf_stations.cx[
    minx - marge_x : maxx + marge_x, 
    miny - marge_y : maxy + marge_y] #.cx fait un slicing spaital qui permet de sélectionner les stations dans la zone d'étude (Saint-Ouen) en utilisant les limites géographiques calculées précédemment

In [ ]:
# Visualisation de la zone d'étude et de la ligne 14, carte de référence qu'on va utiliser pour la suite
fig, ax = plt.subplots(figsize=(10, 8))

zone_etude.plot(
    ax=ax,
    color="#f2f2f2",
    edgecolor="#555555",
    linewidth=0.8,
    label="Quartiers (IRIS)"
)
ligne_14.plot(
    ax=ax,
    color="#662483",
    linewidth=3.5,
    zorder=4,
    label="Ligne 14",
)

stations_st_ouen.plot(
    ax=ax,
    color="white",
    edgecolor="#662483",
    linewidth=2.8,
    markersize=100,
    zorder=3,
    label="Stations de métro"
)

#On ajoute les noms des stations de métro sur la carte
texts = []
for _, row in stations_st_ouen.iterrows():
    pt = row.geometry
    t = ax.text(
        pt.x,
        pt.y,
        row["nom_zda"],
        fontsize=10.5,
        fontweight="semibold",
        color="#1f2421",
        zorder=6,
        bbox=dict(
            boxstyle="round,pad=0.28",
            facecolor="white",
            edgecolor="#d0d5dd",
            linewidth=0.7,
            alpha=0.92,
        ),
    )
    texts.append(t)

# Ajouter une petite marge (ex: 10% autour) pour que les tracés ne touchent pas les bords du cadre

# Appliquer le zoom sur les axes du graphique
ax.set_xlim(minx - marge_x, maxx + marge_x)
ax.set_ylim(miny - marge_y, maxy + marge_y)

#On n'affiche pas les axes pour un rendu plus esthétique
ax.set_axis_off()

plt.show()

In [ ]:
#1- Chargement des données immobilières
